# Cross-View Geolocation — Training Notebook

Train satellite and phone encoders for cross-view geolocation using
the CV-Cities dataset and symmetric InfoNCE loss.

**Runtime:** Google Colab with T4 GPU

**Pipeline:**
1. Clone repo & install dependencies
2. Download CV-Cities dataset (seattle, london, tokyo, sydney)
3. Train ResNet-50 teacher encoders (~12-15 hrs)
4. Distill to MobileNetV3 student (~5 hrs)
5. Export to ONNX + TFLite (int8 quantized)
6. Download trained models

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repo
!git clone https://github.com/farmino1/cross-view-geolocator.git
%cd cross-view-geolocator

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q huggingface_hub onnx onnxruntime onnx-tf

In [ ]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/geolocator'
!mkdir -p $WORK_DIR/checkpoints/teacher
!mkdir -p $WORK_DIR/checkpoints/student
!mkdir -p $WORK_DIR/exported

## 2. Download CV-Cities Dataset

Downloads ~10GB total. Each zip extracts to `data/<city>/satellite/` and `data/<city>/streetview/`.

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile

CITIES = ['seattle', 'london', 'tokyo', 'sydney']
DATA_DIR = 'data'

for city in CITIES:
    print(f'Downloading {city}...')
    zip_path = hf_hub_download(
        repo_id='gaoshuang98/CV-Cities',
        filename=f'{city}.zip',
        repo_type='dataset',
    )
    print(f'  Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print(f'  Done.')

print('\nDataset ready.')
!ls -la $DATA_DIR/

In [ ]:
# Verify directory structure
import os
for city in CITIES:
    sat = len(os.listdir(f'{DATA_DIR}/{city}/satellite'))
    street = len(os.listdir(f'{DATA_DIR}/{city}/streetview'))
    print(f'{city}: {sat} satellite, {street} streetview')

## 3. Train Teacher Encoders (ResNet-50)

Both satellite and phone encoders are ResNet-50 projecting to shared 256-dim space.
Loss: Symmetric InfoNCE with learnable temperature (init 0.07).
Estimated time: ~12-15 hours on T4.

In [ ]:
from src.training.train import train

train(
    data_dir=DATA_DIR,
    cities=CITIES,
    output_dir=f'{WORK_DIR}/checkpoints/teacher',
    epochs=50,
    batch_size=256,
    lr=3e-4,
    weight_decay=0.2,
    embed_dim=256,
    warmup_epochs=1,
    checkpoint_interval=10,
    device='cuda',
)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt
import json

with open(f'{WORK_DIR}/checkpoints/teacher/history.json') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
r1 = [h.get('recall@1', 0) for h in history]
r5 = [h.get('recall@5', 0) for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(epochs, r1, label='R@1')
ax2.plot(epochs, r5, label='R@5')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Recall')
ax2.set_title('Validation Recall@K')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Distill to MobileNetV3

Student (MobileNetV3-Large) learns to mimic the teacher's phone encoder embeddings.
Estimated time: ~5 hours on T4.

In [ ]:
from src.training.distill import distill

distill(
    teacher_checkpoint=f'{WORK_DIR}/checkpoints/teacher/best_model.pt',
    data_dir=DATA_DIR,
    cities=CITIES,
    output_dir=f'{WORK_DIR}/checkpoints/student',
    epochs=30,
    batch_size=256,
    lr=1e-4,
    embed_dim=256,
    device='cuda',
)

## 5. Export to ONNX + TFLite

Exports:
- `satellite_encoder.onnx` — ResNet-50 for desktop tile encoding
- `phone_encoder_teacher.onnx` — ResNet-50 teacher reference
- `phone_encoder.onnx` — MobileNetV3 student for mobile
- `phone_encoder.tflite` — int8 quantized MobileNetV3 for Android

In [ ]:
from src.training.export import export

export(
    checkpoint_path=f'{WORK_DIR}/checkpoints/teacher/best_model.pt',
    output_dir=f'{WORK_DIR}/exported',
    embed_dim=256,
    export_satellite=True,
    export_phone=True,
    quantize_tflite=True,
)

## 6. Download Models

The files you need:
- `phone_encoder.tflite` — goes on the phone
- `satellite_encoder.onnx` — used with `package_area.py` to build tile indices

In [ ]:
!ls -lh $WORK_DIR/exported/

# Copy to Colab root for easy download
!cp $WORK_DIR/exported/phone_encoder.tflite /content/
!cp $WORK_DIR/exported/satellite_encoder.onnx /content/
!cp $WORK_DIR/exported/phone_encoder.onnx /content/

print('\nFiles ready for download:')
print('  phone_encoder.tflite      — MobileNetV3 int8 for Android')
print('  satellite_encoder.onnx    — ResNet-50 for desktop tile encoding')
print('  phone_encoder.onnx        — MobileNetV3 float32 reference')

In [ ]:
# Download files from Colab
from google.colab import files
files.download('/content/phone_encoder.tflite')
files.download('/content/satellite_encoder.onnx')